In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind, t
from mne.stats import permutation_cluster_test

import os
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.paths import ERPAC_DIR
from config.config import COUPLINGS, TASK_STAGES, ROI, TOI

%matplotlib qt

In [ ]:
# ============================================================
# 1. LOAD TIME-RESOLVED ERPAC DATA
# ============================================================

erpac_df = pd.read_parquet(
    os.path.join(
        ERPAC_DIR,
        "erpac_results.parquet"
    )
)

# Expected columns:
#
# sub
# group
# task
# task_stage
# coupling
# roi
# amp_freq
# time
# erpac_value


# ============================================================
# 2. AVERAGE ACROSS GAMMA FREQUENCIES
# ============================================================

erpac_time_df = (
    erpac_df
    .groupby(
        [
            "sub",
            "group",
            "task",
            "task_stage",
            "coupling",
            "roi",
            "time",
        ],
        as_index=False,
    )["erpac_value"]
    .mean()
)

# ============================================================
# 3. STATISTIC FUNCTION
# ============================================================

def independent_t_stat(x, y):
    """
    Independent-samples t statistic at each time point.

    Returns t values only, as required by MNE.
    """

    t_values, _ = ttest_ind(
        x,
        y,
        axis=0,
        equal_var=True,
        nan_policy="omit",
    )

    return t_values

# ============================================================
# 4. CLUSTER PERMUTATION TEST
# ============================================================

def run_time_cluster_test(
    df,
    coupling,
    stage,
    roi,
    task="FTT",
    n_permutations=10000,
    cluster_alpha=0.05,
    seed=42,
):

    # --------------------------------------------------------
    # Select condition
    # --------------------------------------------------------

    df_test = df[
        (df["task"] == task)
        & (df["coupling"] == coupling)
        & (df["task_stage"] == stage)
        & (df["roi"] == roi)
    ].copy()


    # --------------------------------------------------------
    # Restrict analysis interval
    # --------------------------------------------------------

    if stage == "plan":

        df_test = df_test[
            (df_test["time"] >= TOI["plan"]["start"])
            & (df_test["time"] <= TOI["plan"]["end"])
        ]

    elif stage == "go":

        df_test = df_test[
            (df_test["time"] >= TOI["go"]["start"])
            & (df_test["time"] <= TOI["go"]["end"])
        ]


    # --------------------------------------------------------
    # Convert dataframe into subject × time matrix
    # --------------------------------------------------------

    young = (
        df_test[
            df_test["group"] == "Y"
        ]
        .pivot(
            index="sub",
            columns="time",
            values="erpac_value",
        )
        .sort_index(axis=1)
    )

    old = (
        df_test[
            df_test["group"] == "O"
        ]
        .pivot(
            index="sub",
            columns="time",
            values="erpac_value",
        )
        .sort_index(axis=1)
    )


    # --------------------------------------------------------
    # Make sure time vectors are identical
    # --------------------------------------------------------

    common_times = young.columns.intersection(
        old.columns
    )

    young = young[common_times]
    old = old[common_times]

    times = common_times.to_numpy(
        dtype=float
    )


    # --------------------------------------------------------
    # Convert to NumPy arrays
    #
    # shape:
    # subjects × time
    # --------------------------------------------------------

    X_young = young.to_numpy()
    X_old = old.to_numpy()

    print(
        f"\n{coupling} | {stage} | {roi}"
    )

    print(
        f"Young: {X_young.shape}"
    )

    print(
        f"Old:   {X_old.shape}"
    )


    # --------------------------------------------------------
    # Cluster-forming t threshold
    #
    # two-sided alpha = .05
    # --------------------------------------------------------

    dfree = (
        X_young.shape[0]
        + X_old.shape[0]
        - 2
    )

    threshold = t.ppf(
        1 - cluster_alpha / 2,
        df=dfree,
    )


    print(
        f"Cluster-forming threshold: "
        f"|t| > {threshold:.3f}"
    )


    # --------------------------------------------------------
    # Cluster permutation
    # --------------------------------------------------------

    T_obs, clusters, cluster_p_values, H0 = (
        permutation_cluster_test(
            [
                X_young,
                X_old,
            ],
            stat_fun=independent_t_stat,
            threshold=threshold,
            tail=0,
            n_permutations=n_permutations,
            out_type="mask",
            seed=seed,
            n_jobs=-1,
            verbose=False,
        )
    )


    return {
        "coupling": coupling,
        "stage": stage,
        "roi": roi,
        "T_obs": T_obs,
        "clusters": clusters,
        "cluster_p_values": cluster_p_values,
        "H0": H0,
        "times": times,
        "X_young": X_young,
        "X_old": X_old,
    }

test

In [8]:
result = run_time_cluster_test(
    df=erpac_time_df,
    coupling="theta_gamma",
    stage="go",
    roi="M1",
)

result


theta_gamma | go | M1
Young: (22, 326)
Old:   (24, 326)
Cluster-forming threshold: |t| > 2.015


{'T_obs': array([-7.49646544e-01, -7.61794269e-01, -7.88344979e-01, -8.10329795e-01,
        -8.35725725e-01, -8.94758284e-01, -9.78622496e-01, -1.09042943e+00,
        -1.21905875e+00, -1.37029731e+00, -1.52444923e+00, -1.70323110e+00,
        -1.85449517e+00, -1.97656703e+00, -2.01214790e+00, -2.01268315e+00,
        -1.97777712e+00, -1.89658117e+00, -1.78634226e+00, -1.62577260e+00,
        -1.44983935e+00, -1.26389289e+00, -1.07544005e+00, -8.64854574e-01,
        -6.62327588e-01, -4.70259577e-01, -2.67826378e-01, -9.16488767e-02,
         6.71482161e-02,  1.94017306e-01,  2.90165305e-01,  3.81574482e-01,
         4.47551072e-01,  5.18601596e-01,  5.73640704e-01,  6.04123414e-01,
         6.19467616e-01,  5.98401666e-01,  5.58726549e-01,  5.11887074e-01,
         4.61001217e-01,  3.96076113e-01,  3.12303126e-01,  2.26083666e-01,
         1.45551682e-01,  6.26208708e-02, -5.34009980e-03, -7.26012588e-02,
        -1.35908678e-01, -2.20164999e-01, -3.09936017e-01, -3.91549051e-01,
   

In [ ]:
# ============================================================
# 5. PRINT SIGNIFICANT CLUSTERS
# ============================================================

for i, (cluster, p_value) in enumerate(
    zip(
        result["clusters"],
        result["cluster_p_values"],
    )
):

    if p_value < 0.05:

        cluster_times = result["times"][
            cluster
        ]

        print(
            f"Condition: {result['coupling']} | {result['stage']} | {result['roi']}"
            f"Cluster {i}: "
            f"{cluster_times[0]:.3f} – "
            f"{cluster_times[-1]:.3f} s, "
            f"p = {p_value:.4f}"
        )


In [20]:
# ============================================================
# PLOT CLUSTER RESULTS
# ============================================================

def plot_cluster_result(
    result,
    coupling,
    stage,
    roi,
    alpha=0.05,
):

    times = result["times"]

    X_young = result["X_young"]
    X_old = result["X_old"]


    # --------------------------------------------------------
    # Group means
    # --------------------------------------------------------

    young_mean = X_young.mean(axis=0)
    old_mean = X_old.mean(axis=0)


    # --------------------------------------------------------
    # SEM
    # --------------------------------------------------------

    young_sem = (
        X_young.std(
            axis=0,
            ddof=1
        )
        / np.sqrt(
            X_young.shape[0]
        )
    )

    old_sem = (
        X_old.std(
            axis=0,
            ddof=1
        )
        / np.sqrt(
            X_old.shape[0]
        )
    )


    # --------------------------------------------------------
    # Figure
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(8, 4.5)
    )


    # --------------------------------------------------------
    # Mean ERPAC
    # --------------------------------------------------------

    ax.plot(
        times,
        young_mean,
        label="Young",
        linewidth=2,
        color="#4C78A8",
    )

    ax.plot(
        times,
        old_mean,
        label="Older",
        linewidth=2,
        color="#E07B53",
    )


    # --------------------------------------------------------
    # SEM shading
    # --------------------------------------------------------

    ax.fill_between(
        times,
        young_mean - young_sem,
        young_mean + young_sem,
        color="#4C78A8",
        alpha=0.18,
        linewidth=0,
    )

    ax.fill_between(
        times,
        old_mean - old_sem,
        old_mean + old_sem,
        color="#E07B53",
        alpha=0.18,
        linewidth=0,
    )


    # --------------------------------------------------------
    # Determine y limits before plotting cluster bars
    # --------------------------------------------------------

    y_min = min(
        (young_mean - young_sem).min(),
        (old_mean - old_sem).min(),
    )

    y_max = max(
        (young_mean + young_sem).max(),
        (old_mean + old_sem).max(),
    )

    y_range = y_max - y_min

    ax.set_ylim(
        y_min - 0.10 * y_range,
        y_max + 0.05 * y_range,
    )


    # --------------------------------------------------------
    # Significant clusters
    # --------------------------------------------------------

    significant_found = False

    for i, (cluster, p_value) in enumerate(
        zip(
            result["clusters"],
            result["cluster_p_values"],
        )
    ):

        if p_value < alpha:

            significant_found = True

            # ------------------------------------------------
            # Convert MNE cluster representation to indices
            # ------------------------------------------------

            # MNE often returns a tuple for each cluster
            if isinstance(cluster, tuple):
                cluster = cluster[0]

            # Boolean mask
            if isinstance(cluster, np.ndarray) and cluster.dtype == bool:
                cluster_idx = np.where(cluster)[0]

            # Integer indices
            elif isinstance(cluster, np.ndarray):
                cluster_idx = cluster

            # Slice
            elif isinstance(cluster, slice):
                cluster_idx = np.arange(
                    len(times)
                )[cluster]

            else:
                raise TypeError(
                    f"Unexpected cluster type: {type(cluster)}"
                )


            # ------------------------------------------------
            # Get cluster start/end times
            # ------------------------------------------------

            cluster_times = times[
                cluster_idx
            ]

            t_start = cluster_times[0]
            t_end = cluster_times[-1]


            # ------------------------------------------------
            # Shade significant temporal interval
            # ------------------------------------------------

            ax.axvspan(
                t_start,
                t_end,
                color="grey",
                alpha=0.20,
                linewidth=0,
                zorder=0,
            )


            # ------------------------------------------------
            # Add significance bar at bottom
            # ------------------------------------------------

            cluster_y = (
                y_min
                - 0.055 * y_range
            )

            ax.plot(
                [
                    t_start,
                    t_end,
                ],
                [
                    cluster_y,
                    cluster_y,
                ],
                color="black",
                linewidth=4,
                solid_capstyle="butt",
                zorder=20,
            )


            # ------------------------------------------------
            # Print cluster info
            # ------------------------------------------------

            print(
                f"{coupling} | {stage} | {roi} | "
                f"Cluster {i}: "
                f"{t_start:.3f}–{t_end:.3f} s, "
                f"p = {p_value:.4f}"
            )


    # --------------------------------------------------------
    # If no significant clusters
    # --------------------------------------------------------

    if not significant_found:

        print(
            f"{coupling} | {stage} | {roi}: "
            "No significant clusters."
        )


    # --------------------------------------------------------
    # Event marker
    # --------------------------------------------------------

    ax.axvline(
        0,
        color="black",
        linestyle="--",
        linewidth=1,
        alpha=0.5,
    )


    # --------------------------------------------------------
    # Formatting
    # --------------------------------------------------------

    coupling_label = (
        coupling
        .replace("theta_gamma", "θ–γ")
        .replace("alpha_gamma", "α–γ")
        .replace("beta_gamma", "β–γ")
    )

    stage_label = {
        "plan": "Planning",
        "go": "Execution",
    }.get(stage, stage)


    ax.set_xlabel(
        "Time (s)"
    )

    ax.set_ylabel(
        "ERPAC"
    )

    ax.set_title(
        f"{coupling_label} | {stage_label} | {roi}",
        fontweight="bold",
    )

    ax.legend(
        frameon=False
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.grid(
        axis="y",
        alpha=0.15,
    )

    plt.tight_layout()

    return fig

Check all conditions

In [ ]:
# ============================================================
# RUN ALL CONDITIONS + SAVE RESULTS TO TXT
# ============================================================

output_file = os.path.join(
    ERPAC_DIR,
    "cluster_permutation_results_test.txt"
)

cluster_results = {}


with open(output_file, "w") as f:

    for coupling in COUPLINGS.keys():

        for stage in TASK_STAGES:

            for roi in ROI.keys():

                header = (
                    "\n"
                    + "=" * 60
                    + "\n"
                    + f"Condition: {coupling} | {stage} | {roi}\n"
                    + "=" * 60
                    + "\n"
                )

                # Print to console
                print(header)

                # Save to txt
                f.write(header)


                # --------------------------------------------
                # Run cluster test
                # --------------------------------------------

                result = run_time_cluster_test(
                    df=erpac_time_df,
                    coupling=coupling,
                    stage=stage,
                    roi=roi,
                )


                # Store result
                key = (
                    coupling,
                    stage,
                    roi,
                )

                cluster_results[key] = result


                # --------------------------------------------
                # Significant clusters
                # --------------------------------------------

                significant_found = False

                for i, (cluster, p_value) in enumerate(
                    zip(
                        result["clusters"],
                        result["cluster_p_values"],
                    )
                ):

                    if p_value < 0.05:

                        significant_found = True

                        cluster_times = result["times"][
                            cluster
                        ]

                        line = (
                            f"Cluster {i}: "
                            f"{cluster_times[0]:.3f} – "
                            f"{cluster_times[-1]:.3f} s, "
                            f"p = {p_value:.4f}\n"
                        )

                        # Console
                        print(line, end="")

                        # Txt file
                        f.write(line)

                        fig = plot_cluster_result(
                                result,
                                coupling=coupling,
                                stage=stage,
                                roi=roi,
                            )

                        plt.show()


                # --------------------------------------------
                # No significant clusters
                # --------------------------------------------

                if not significant_found:

                    line = "No significant clusters.\n"

                    print(line, end="")
                    f.write(line)


print(
    f"\nResults saved to:\n{output_file}"
)


Condition: theta_gamma | plan | M1


theta_gamma | plan | M1
Young: (23, 251)
Old:   (24, 251)
Cluster-forming threshold: |t| > 2.014
No significant clusters.

Condition: theta_gamma | plan | S1


theta_gamma | plan | S1
Young: (23, 251)
Old:   (24, 251)
Cluster-forming threshold: |t| > 2.014
No significant clusters.

Condition: theta_gamma | plan | PMC


theta_gamma | plan | PMC
Young: (23, 251)
Old:   (24, 251)
Cluster-forming threshold: |t| > 2.014
No significant clusters.

Condition: theta_gamma | plan | SMA


theta_gamma | plan | SMA
Young: (23, 251)
Old:   (24, 251)
Cluster-forming threshold: |t| > 2.014
Cluster 2: 0.286 – 0.338 s, p = 0.0398

Condition: theta_gamma | go | M1


theta_gamma | go | M1
Young: (23, 326)
Old:   (24, 326)
Cluster-forming threshold: |t| > 2.014
No significant clusters.

Condition: theta_gamma | go | S1


theta_gamma | go | S1
Young: (23, 326)
Old:   (24, 326)
Cluster-forming threshold: |t| > 2.014
No significant clusters.

Condition: theta_gamma | go |

C:\Users\a1902989\AppData\Local\Temp\ipykernel_28640\179749935.py:233: RuntimeWarning: No clusters found, returning empty H0, clusters, and cluster_pv
  permutation_cluster_test(


No significant clusters.

Condition: alpha_gamma | plan | SMA


alpha_gamma | plan | SMA
Young: (23, 251)
Old:   (24, 251)
Cluster-forming threshold: |t| > 2.014
No significant clusters.

Condition: alpha_gamma | go | M1


alpha_gamma | go | M1
Young: (23, 326)
Old:   (24, 326)
Cluster-forming threshold: |t| > 2.014


C:\Users\a1902989\AppData\Local\Temp\ipykernel_28640\179749935.py:233: RuntimeWarning: No clusters found, returning empty H0, clusters, and cluster_pv
  permutation_cluster_test(
C:\Users\a1902989\AppData\Local\Temp\ipykernel_28640\179749935.py:233: RuntimeWarning: No clusters found, returning empty H0, clusters, and cluster_pv
  permutation_cluster_test(


No significant clusters.

Condition: alpha_gamma | go | S1


alpha_gamma | go | S1
Young: (23, 326)
Old:   (24, 326)
Cluster-forming threshold: |t| > 2.014
No significant clusters.

Condition: alpha_gamma | go | PMC


alpha_gamma | go | PMC
Young: (23, 326)
Old:   (24, 326)
Cluster-forming threshold: |t| > 2.014
No significant clusters.

Condition: alpha_gamma | go | SMA


alpha_gamma | go | SMA
Young: (23, 326)
Old:   (24, 326)
Cluster-forming threshold: |t| > 2.014
No significant clusters.

Condition: beta_gamma | plan | M1


beta_gamma | plan | M1
Young: (23, 251)
Old:   (24, 251)
Cluster-forming threshold: |t| > 2.014
No significant clusters.

Condition: beta_gamma | plan | S1


beta_gamma | plan | S1
Young: (23, 251)
Old:   (24, 251)
Cluster-forming threshold: |t| > 2.014


C:\Users\a1902989\AppData\Local\Temp\ipykernel_28640\179749935.py:233: RuntimeWarning: No clusters found, returning empty H0, clusters, and cluster_pv
  permutation_cluster_test(


No significant clusters.

Condition: beta_gamma | plan | PMC


beta_gamma | plan | PMC
Young: (23, 251)
Old:   (24, 251)
Cluster-forming threshold: |t| > 2.014
No significant clusters.

Condition: beta_gamma | plan | SMA


beta_gamma | plan | SMA
Young: (23, 251)
Old:   (24, 251)
Cluster-forming threshold: |t| > 2.014
Cluster 0: 0.106 – 0.148 s, p = 0.0495

Condition: beta_gamma | go | M1


beta_gamma | go | M1
Young: (23, 326)
Old:   (24, 326)
Cluster-forming threshold: |t| > 2.014
No significant clusters.

Condition: beta_gamma | go | S1


beta_gamma | go | S1
Young: (23, 326)
Old:   (24, 326)
Cluster-forming threshold: |t| > 2.014
No significant clusters.

Condition: beta_gamma | go | PMC


beta_gamma | go | PMC
Young: (23, 326)
Old:   (24, 326)
Cluster-forming threshold: |t| > 2.014
No significant clusters.

Condition: beta_gamma | go | SMA


beta_gamma | go | SMA
Young: (23, 326)
Old:   (24, 326)
Cluster-forming threshold: |t| > 2.014
No significant clusters.


C:\Users\a1902989\AppData\Local\Temp\ipykernel_28640\179749935.py:233: RuntimeWarning: No clusters found, returning empty H0, clusters, and cluster_pv
  permutation_cluster_test(



Results saved to:
F:\# study 2\eeg_data\erpac\cluster_permutation_results_test.txt


NOTE: the figures show exploratory temporal clusters, not corrected significant age effects

Bonferroni across 24 tests:

theta–gamma SMA: 0.0371 × 24 = 0.8904;
beta–gamma SMA:  0.0459 × 24 > 1

_______________________

YOUNGER GROUP

In [ ]:
erpac_df = pd.read_parquet(
    os.path.join(
        ERPAC_DIR,
        "erpac_results.parquet"
    )
)
